Step 9 — KPI Framework
Nassau Candy Distributor: Factory Reallocation & Shipping Optimization

In [3]:
# Part A — Set up Step 9
# Cell 1: Import libraries and configure paths


import os
import numpy as np
import pandas as pd

PROJECT_DIR =  r"nassau-candy-factory-optimization\data"


STEP7_DIR = os.path.join(
    PROJECT_DIR,
    "step7_outputs"
)

STEP8_DIR = os.path.join(
    PROJECT_DIR,
    "step8_outputs"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "step9_outputs"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Step 7 folder exists:", os.path.exists(STEP7_DIR))
print("Step 8 folder exists:", os.path.exists(STEP8_DIR))
print("Step 9 output folder:", OUTPUT_DIR)


Project folder: nassau-candy-factory-optimization\data
Step 7 folder exists: True
Step 8 folder exists: True
Step 9 output folder: nassau-candy-factory-optimization\data\step9_outputs


In [4]:
# Part B — Load the existing analysis results
# Cell 2: Load Step 7 and Step 8 outputs
product_file = os.path.join(
    STEP7_DIR,
    "product_financial_risk_summary.csv"
)

region_file = os.path.join(
    STEP7_DIR,
    "region_financial_risk_summary.csv"
)

shipping_risk_file = os.path.join(
    STEP7_DIR,
    "shipping_operational_risk.csv"
)

optimization_readiness_file = os.path.join(
    STEP8_DIR,
    "optimization_readiness_report.csv"
)

required_files = [
    product_file,
    region_file,
    shipping_risk_file,
    optimization_readiness_file
]

for file_path in required_files:
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Required output not found: {file_path}. "
            "Complete Step 7 and Step 8 first."
        )

product_summary = pd.read_csv(product_file)
region_summary = pd.read_csv(region_file)
shipping_risk = pd.read_csv(shipping_risk_file)
optimization_readiness = pd.read_csv(
    optimization_readiness_file
)

print("Product summary:", product_summary.shape)
print("Region summary:", region_summary.shape)
print("Shipping risk summary:", shipping_risk.shape)
print("Optimization readiness:", optimization_readiness.shape)


Product summary: (15, 10)
Region summary: (4, 9)
Shipping risk summary: (16, 8)
Optimization readiness: (7, 3)


In [7]:
# Part C — KPI 1: Lead Time Reduction %

# Lead Time Reduction measures how much shipping lead time changes between a baseline and an alternative scenario.

# $$ \text{Lead Time Reduction (\%)} = \frac{\text{Baseline Lead Time}-\text{Scenario Lead Time}} {\text{Baseline Lead Time}}\times100 $$

# For example, suppose a validated baseline is 10 days and a validated alternative scenario is 8 days.

# $$ \frac{10-8}{10}\times100=20\% $$


# Cell 3: Create a reusable function
def calculate_lead_time_reduction(
    baseline_days,
    scenario_days
):
    """
    Calculate lead-time reduction percentage.

    Positive result: lead time decreased.
    Zero: no change.
    Negative result: lead time increased.
    """

    if pd.isna(baseline_days) or pd.isna(scenario_days):
        return np.nan

    if baseline_days <= 0:
        return np.nan

    if scenario_days < 0:
        return np.nan

    return (
        (baseline_days - scenario_days)
        / baseline_days
    ) * 100


# Illustrative calculation only
example_reduction = calculate_lead_time_reduction(
    baseline_days=10,
    scenario_days=8
)

print(
    f"Illustrative lead-time reduction: "
    f"{example_reduction:.2f}%"
)


Illustrative lead-time reduction: 20.00%


In [10]:
# Cell 4: Calculate the KPI when real scenario data is available


def create_lead_time_kpi(scenario_df):

    required = [
        "Product ID",
        "Baseline Lead Time Days",
        "Scenario Lead Time Days"
    ]

    missing = [
        col for col in required
        if col not in scenario_df.columns
    ]

    if missing:
        raise ValueError(
            "Missing columns: " + ", ".join(missing)
        )

    result = scenario_df.copy()

    result["Lead Time Reduction (%)"] = result.apply(
        lambda row: calculate_lead_time_reduction(
            row["Baseline Lead Time Days"],
            row["Scenario Lead Time Days"]
        ),
        axis=1
    )

    return result

#Current status: Your actual factory-reallocation lead-time reduction remains unavailable until factory-specific predictions or measured comparisons are available.


In [11]:
# Part D — KPI 2: Profit Impact Stability

#The KPI for this project should measure how consistent the estimated financial impact of a reallocation scenario is across comparable observations, repeated evaluations, or uncertainty simulations.

#For valid scenario-level profit impacts:

# $$ \text{Profit Impact CV} = \frac{\text{Standard Deviation of Profit Impact}} {|\text{Mean Profit Impact}|} $$

# A lower coefficient of variation generally indicates more stable estimates, provided the mean is sufficiently far from zero and the observations are comparable.

# Cell 5: Create the profit stability function
def calculate_profit_impact_stability(
    profit_impacts,
    minimum_observations=2
):
    """
    Calculate the coefficient of variation for scenario-level
    incremental profit impacts.

    This function requires actual, comparable scenario impacts.
    It does not calculate incremental profit from historical
    gross profit alone.
    """

    values = pd.to_numeric(
        pd.Series(profit_impacts),
        errors="coerce"
    ).dropna()

    if len(values) < minimum_observations:
        return {
            "Observation Count": len(values),
            "Mean Profit Impact": values.mean()
            if len(values) else np.nan,
            "Profit Impact Std": np.nan,
            "Profit Impact CV": np.nan,
            "Status": "Insufficient observations"
        }

    mean_impact = values.mean()
    std_impact = values.std(ddof=1)

    if abs(mean_impact) < 1e-9:
        cv = np.nan
        status = "Mean near zero; CV not reliable"
    else:
        cv = std_impact / abs(mean_impact)
        status = "Calculated"

    return {
        "Observation Count": len(values),
        "Mean Profit Impact": mean_impact,
        "Profit Impact Std": std_impact,
        "Profit Impact CV": cv,
        "Status": status
    }


print("Profit impact stability function created.")



Profit impact stability function created.


In [12]:
# Part E — KPI 3: Scenario Confidence Score

# A confidence score should not be an arbitrary label. It should reflect whether the scenario has the evidence needed to support a decision.


#The score checks whether key evidence is available:

#Validated current factory assignment
#Factory-aware lead-time prediction
#Validated financial-impact estimate
#Operational feasibility assessment
#Prediction uncertainty assessment



#$$ \text{Evidence Readiness Score} = \frac{\text{Completed Evidence Checks}}{5}\times100 $$

#This is a project-defined completeness score, not a probability that a recommendation is correct.

#Cell 6: Create the evidence-readiness scoring function
def calculate_evidence_readiness(
    current_factory_validated,
    factory_aware_prediction_available,
    financial_impact_validated,
    operational_feasibility_validated,
    prediction_uncertainty_assessed
):
    checks = {
        "Current factory validated": current_factory_validated,
        "Factory-aware prediction available":
            factory_aware_prediction_available,
        "Financial impact validated": financial_impact_validated,
        "Operational feasibility validated":
            operational_feasibility_validated,
        "Prediction uncertainty assessed":
            prediction_uncertainty_assessed
    }

    completed = sum(
        bool(value) for value in checks.values()
    )

    score = completed / len(checks) * 100

    return {
        "Evidence Readiness Score": score,
        "Completed Checks": completed,
        "Total Checks": len(checks),
        "Missing Evidence": [
            name for name, available in checks.items()
            if not available
        ]
    }


# Current known project status
current_readiness = calculate_evidence_readiness(
    current_factory_validated=False,
    factory_aware_prediction_available=False,
    financial_impact_validated=False,
    operational_feasibility_validated=False,
    prediction_uncertainty_assessed=False
)

print(current_readiness)

#The False values reflect the limitations identified so far in our project workflow. Update them only when you have verified evidence.

#A low readiness score means the scenario requires more supporting data. It does not mean the factory itself performs poorly.


{'Evidence Readiness Score': 0.0, 'Completed Checks': 0, 'Total Checks': 5, 'Missing Evidence': ['Current factory validated', 'Factory-aware prediction available', 'Financial impact validated', 'Operational feasibility validated', 'Prediction uncertainty assessed']}


In [14]:
# Part F — KPI 4: Recommendation Coverage %

# $$ \text{Recommendation Coverage (\%)} = \frac{\text{Products with Valid Recommendations}} {\text{Eligible Products}}\times100 $$

#Cell 7: Create the coverage function
def calculate_recommendation_coverage(
    eligible_products,
    recommended_products
):
    if eligible_products < 0 or recommended_products < 0:
        raise ValueError("Counts cannot be negative.")

    if recommended_products > eligible_products:
        raise ValueError(
            "Recommended products cannot exceed eligible products."
        )

    if eligible_products == 0:
        coverage = np.nan
    else:
        coverage = (
            recommended_products / eligible_products
        ) * 100

    return {
        "Eligible Products": eligible_products,
        "Products with Valid Recommendations":
            recommended_products,
        "Recommendation Coverage (%)": coverage
    }


print("Recommendation coverage function created.")


Recommendation coverage function created.


In [15]:
# Cell 8: Calculate current mapping coverage


MAPPING_FILE = os.path.join(
    PROJECT_DIR,
    "product_factory_mapping.csv"
)

if not os.path.exists(MAPPING_FILE):
    raise FileNotFoundError(
        f"Mapping file not found: {MAPPING_FILE}"
    )

mapping = pd.read_csv(MAPPING_FILE)

required_mapping_columns = [
    "Product ID",
    "Current Factory"
]

missing_columns = [
    col for col in required_mapping_columns
    if col not in mapping.columns
]

if missing_columns:
    raise ValueError(
        "Mapping file is missing columns: "
        + ", ".join(missing_columns)
    )

mapping["Current Factory"] = (
    mapping["Current Factory"]
    .astype("string")
    .str.strip()
)

valid_factories = {
    "Lot's O' Nuts",
    "Wicked Choccy's",
    "Sugar Shack",
    "Secret Factory",
    "The Other Factory"
}

eligible_products = mapping["Product ID"].nunique()

valid_mapping = mapping.loc[
    mapping["Current Factory"].isin(valid_factories)
].copy()

valid_mapping_products = valid_mapping[
    "Product ID"
].nunique()

mapping_coverage = calculate_recommendation_coverage(
    eligible_products=eligible_products,
    recommended_products=valid_mapping_products
)

print("Current factory mapping coverage:")
print(mapping_coverage)

#Important: This measures the percentage of products with a recognized factory assignment, not the percentage with a valid reallocation recommendation. It is a prerequisite coverage KPI.


Current factory mapping coverage:
{'Eligible Products': 15, 'Products with Valid Recommendations': 15, 'Recommendation Coverage (%)': 100.0}


In [16]:
# Part G — KPI 5: Operational Risk Indicator


#The coefficient of variation is:

#$$ CV_{\text{shipping}} = \frac{\text{Standard Deviation of Lead Time}} {\text{Mean Lead Time}} $$

#It is only meaningful when the mean is positive and enough valid observations exist.

#Cell 9: Calculate historical shipping variability
if "Shipping_Lead_Time_Std" not in product_summary.columns:
    raise ValueError(
        "Shipping_Lead_Time_Std is missing from the "
        "Step 7 product summary."
    )

if "Average_Shipping_Lead_Time" not in product_summary.columns:
    raise ValueError(
        "Average_Shipping_Lead_Time is missing from the "
        "Step 7 product summary."
    )

if "Order_Lines" not in product_summary.columns:
    raise ValueError(
        "Order_Lines is missing from the Step 7 product summary."
    )

risk_kpi = product_summary[
    [
        "Product ID",
        "Order_Lines",
        "Average_Shipping_Lead_Time",
        "Shipping_Lead_Time_Std"
    ]
].copy()

risk_kpi["Shipping Lead Time CV"] = np.where(
    (
        (risk_kpi["Order_Lines"] >= 2)
        & (risk_kpi["Average_Shipping_Lead_Time"] > 0)
    ),
    (
        risk_kpi["Shipping_Lead_Time_Std"]
        / risk_kpi["Average_Shipping_Lead_Time"]
    ),
    np.nan
)

risk_kpi["Risk Indicator Status"] = np.where(
    risk_kpi["Shipping Lead Time CV"].notna(),
    "Calculated from historical variability",
    "Insufficient data or nonpositive mean"
)

display(
    risk_kpi.sort_values(
        "Shipping Lead Time CV",
        ascending=False,
        na_position="last"
    ).head(15)
)

risk_kpi.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "historical_shipping_risk_kpi.csv"
    ),
    index=False
)

#A higher shipping-time CV indicates greater variability relative to the mean. It is not a complete operational risk score and does not establish the cause of that variability.


,Product ID,Order_Lines,Average_Shipping_Lead_Time,Shipping_Lead_Time_Std,Shipping Lead Time CV,Risk Indicator Status
13,SUG-NER-92000,4,1274.000000,422.043442,0.331274,Calculated from historical variability
10,OTH-FIZ-56000,6,1273.500000,326.691751,0.256531,Calculated from historical variability
6,OTH-GUM-21000,120,1306.525000,292.031596,0.223518,Calculated from historical variability
5,OTH-LIC-15000,94,1339.138298,288.459855,0.215407,Calculated from historical variability
8,OTH-KAZ-38000,96,1272.989583,259.943212,0.204199,Calculated from historical variability
4,CHO-FUD-51000,1818,1314.242024,264.102825,0.200954,Calculated from historical variability
2,CHO-MIL-31000,2137,1316.764623,263.692704,0.200258,Calculated from historical variability
0,CHO-SCR-58000,2064,1320.641957,260.121265,0.196966,Calculated from historical variability
1,CHO-TRI-54000,2015,1325.661042,260.524339,0.196524,Calculated from historical variability
3,CHO-NUT-13000,1810,1328.915470,260.837398,0.196278,Calculated from historical variability


In [17]:
#Part H — KPI 6: Gross Profit Margin %

#This KPI can be calculated from historical sales and gross profit, assuming the financial fields are consistent.

#Cell 10: Create a financial KPI table
required_financial_columns = [
    "Total_Sales",
    "Total_Gross_Profit"
]

missing_financial_columns = [
    col for col in required_financial_columns
    if col not in product_summary.columns
]

if missing_financial_columns:
    raise ValueError(
        "Missing financial columns: "
        + ", ".join(missing_financial_columns)
    )

financial_kpi = product_summary[
    [
        "Product ID",
        "Total_Sales",
        "Total_Gross_Profit"
    ]
].copy()

financial_kpi["Historical Gross Profit Margin (%)"] = np.where(
    financial_kpi["Total_Sales"] != 0,
    (
        financial_kpi["Total_Gross_Profit"]
        / financial_kpi["Total_Sales"]
    ) * 100,
    np.nan
)

display(financial_kpi.head(15))

financial_kpi.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "historical_profitability_kpi.csv"
    ),
    index=False
)

#This KPI measures historical profitability. It does not measure the financial effect of moving a product to another factory.

,Product ID,Total_Sales,Total_Gross_Profit,Historical Gross Profit Margin (%)
0,CHO-SCR-58000,27874.80,19357.50,69.444444
1,CHO-TRI-54000,28485.00,18610.20,65.333333
2,CHO-MIL-31000,26867.75,17443.37,64.923077
3,CHO-NUT-13000,23574.95,16819.95,71.346705
4,CHO-FUD-51000,24890.40,16593.60,66.666667
5,OTH-LIC-15000,7860.00,3930.00,50.000000
6,OTH-GUM-21000,597.50,310.70,52.000000
7,SUG-EVE-47000,130.00,104.00,80.000000
8,OTH-KAZ-38000,1205.75,92.75,7.692308
9,SUG-HAI-55000,76.50,59.50,77.777778


In [24]:
# Part I — KPI 7: Optimization Benefit Score

# Once we have reliable scenario-level values, we can combine shipping, profitability, and risk scores into one decision metric.

# $$ \text{Optimization Benefit Score} = w_sS_s+w_pS_p+w_r(100-S_r) $$

# The weights must be approved by the business. The individual scores must be calculated consistently from validated scenario data.

# Cell 11: Create a reusable optimization KPI function
def calculate_optimization_benefit(
    speed_score,
    profit_score,
    risk_score,
    speed_weight=1/3,
    profit_weight=1/3,
    risk_weight=1/3
):
    """
    Calculate a normalized optimization benefit score.

    Scores must be between 0 and 100.
    Higher speed and profit scores are favorable.
    Higher risk scores are unfavorable.
    """

    scores = {
        "speed_score": speed_score,
        "profit_score": profit_score,
        "risk_score": risk_score
    }

    for name, value in scores.items():
        if pd.isna(value) or not np.isfinite(value):
            raise ValueError(
                f"{name} must be a finite numeric value."
            )

        if not 0 <= value <= 100:
            raise ValueError(
                f"{name} must be between 0 and 100."
            )

    weights = [
        speed_weight,
        profit_weight,
        risk_weight
    ]

    if any(weight < 0 for weight in weights):
        raise ValueError("Weights cannot be negative.")

    if not np.isclose(sum(weights), 1.0):
        raise ValueError("Weights must sum to 1.")

    return (
        speed_weight * speed_score
        + profit_weight * profit_score
        + risk_weight * (100 - risk_score)
    )



# print("Optimization Benefit Score function created.")

# Test the function with sample scores
sample_score = calculate_optimization_benefit(
    speed_score=80, 
    profit_score=90, 
    risk_score=20
)
print(f"Sample Optimization Benefit Score: {sample_score:.2f}")

#We will not populate real factory recommendations with this score until the scenario-specific inputs have been validated.


Sample Optimization Benefit Score: 83.33


In [21]:
#Part J — Create the KPI status dashboard
#Cell 12: Summarize KPI availability

#This table indicates which KPIs can currently be calculated and which need more evidence.

kpi_status = pd.DataFrame([
    {
        "KPI": "Lead Time Reduction (%)",
        "Status": "Pending",
        "Reason": "Comparable factory-specific scenario lead times unavailable"
    },
    {
        "KPI": "Profit Impact Stability",
        "Status": "Pending",
        "Reason": "Incremental scenario profit estimates unavailable"
    },
    {
        "KPI": "Scenario Evidence Readiness Score",
        "Status": "Available",
        "Reason": "Can score the completeness of supporting evidence"
    },
    {
        "KPI": "Factory Mapping Coverage (%)",
        "Status": "Available",
        "Reason": "Can measure products with valid factory assignments"
    },
    {
        "KPI": "Recommendation Coverage (%)",
        "Status": "Pending",
        "Reason": "Requires validated recommendations"
    },
    {
        "KPI": "Historical Shipping Variability",
        "Status": "Available",
        "Reason": "Calculated from historical shipping lead times"
    },
    {
        "KPI": "Historical Gross Profit Margin (%)",
        "Status": "Available",
        "Reason": "Calculated from historical sales and gross profit"
    },
    {
        "KPI": "Optimization Benefit Score",
        "Status": "Framework ready",
        "Reason": "Requires validated scenario component scores"
    }
])

display(kpi_status)

kpi_status.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "kpi_availability_status.csv"
    ),
    index=False
)

,KPI,Status,Reason
0,Lead Time Reduction (%),Pending,Comparable factory-specific scenario lead time...
1,Profit Impact Stability,Pending,Incremental scenario profit estimates unavailable
2,Scenario Evidence Readiness Score,Available,Can score the completeness of supporting evidence
3,Factory Mapping Coverage (%),Available,Can measure products with valid factory assign...
4,Recommendation Coverage (%),Pending,Requires validated recommendations
5,Historical Shipping Variability,Available,Calculated from historical shipping lead times
6,Historical Gross Profit Margin (%),Available,Calculated from historical sales and gross profit
7,Optimization Benefit Score,Framework ready,Requires validated scenario component scores


In [22]:
#Part K — Save the Step 9 summary
#Cell 13: Export the methodology
summary_text = """
NASSAU CANDY DISTRIBUTOR
STEP 9 - KPI FRAMEWORK

KPIs defined:
1. Lead Time Reduction Percentage
2. Profit Impact Stability
3. Scenario Evidence Readiness Score
4. Recommendation Coverage Percentage
5. Historical Shipping Variability
6. Historical Gross Profit Margin
7. Optimization Benefit Score

Important distinctions:
- Historical profitability is not incremental scenario profit.
- Historical shipping variability is not factory-specific operational risk.
- Evidence readiness is not a probability of recommendation correctness.
- Factory mapping coverage is not recommendation coverage.
- The baseline shipping model is not factory-aware.

Pending evidence:
- Validated current factory assignments
- Factory-specific lead-time predictions
- Scenario-level incremental financial impacts
- Operational feasibility and capacity information
- Validated prediction uncertainty

No factory reallocation benefit is claimed by this framework.
"""

summary_path = os.path.join(
    OUTPUT_DIR,
    "step9_summary.txt"
)

with open(summary_path, "w", encoding="utf-8") as file:
    file.write(summary_text.strip())

print("Step 9 KPI framework completed.")
print("Output folder:", OUTPUT_DIR)

print("\nFiles created:")

for filename in os.listdir(OUTPUT_DIR):
    print("-", filename)

Step 9 KPI framework completed.
Output folder: nassau-candy-factory-optimization\data\step9_outputs

Files created:
- historical_profitability_kpi.csv
- historical_shipping_risk_kpi.csv
- kpi_availability_status.csv
- step9_summary.txt
